# A1: chạy trên JupyterHub (in-kernel)

Notebook gọi trực tiếp `src.train.run_one` trong kernel: không `subprocess`, không pipe log, không chạy CLI con.

- Chạy lần lượt các cell 1 → 5. Muốn dừng: **Interrupt kernel** (■). Checkpoint ghi kiểu atomic nên `last.pt` của epoch trước vẫn an toàn; chạy lại cell 5 là resume.
- Run có `metrics.json` được bỏ qua; run dở resume từ `last.pt`.
- Không tự rút epoch so với protocol A1.1.

## 1. Cấu hình (sửa ở đây)

In [ ]:
from pathlib import Path
import os, sys

# Notebook mở tại A1/notebooks thì tự nhận A1; nếu không, sửa A1_DIR.
A1_DIR = Path.cwd().parent if (Path.cwd().parent / 'src' / 'train.py').is_file() else Path.home() / 'SDH_HK261_CO5085' / 'A1'
DATA_ROOT = Path.home() / 'datasets'      # chứa food-101/ (đổi sang /scratch/... nếu có)
FOOD_ARCHIVE = None                       # ví dụ Path('/scratch/food-101.tar.gz') nếu đã upload sẵn
OUT = A1_DIR / 'runs'
SPLITS = A1_DIR / 'splits'
HOURS = 11                                # ngân sách mềm; dừng ở ranh giới epoch
NUM_WORKERS = None                        # None = tự chọn theo số CPU và /dev/shm

assert (A1_DIR / 'src' / 'train.py').is_file(), f'Không thấy code A1 tại {A1_DIR}'
os.chdir(A1_DIR)
if str(A1_DIR) not in sys.path:
    sys.path.insert(0, str(A1_DIR))
print('A1:', A1_DIR)

## 2. Môi trường
Chỉ cài gói còn thiếu; không đụng tới torch/CUDA của server.

In [ ]:
import importlib.util
missing = [p for p, m in [('timm', 'timm'), ('pyyaml', 'yaml'), ('tabulate', 'tabulate'),
                          ('scikit-learn', 'sklearn'), ('pandas', 'pandas'), ('matplotlib', 'matplotlib')]
           if importlib.util.find_spec(m) is None]
if missing:
    %pip install -q {' '.join(missing)}

import shutil, torch
assert torch.cuda.is_available(), 'Chọn kernel/profile có GPU'
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0),
      f'| {torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB')

# DataLoader worker truyền batch qua /dev/shm; container hay chỉ có 64 MB -> "Bus error"/worker chết.
shm_gb = shutil.disk_usage('/dev/shm').total / 2**30 if os.path.exists('/dev/shm') else 0
cpus = len(os.sched_getaffinity(0)) if hasattr(os, 'sched_getaffinity') else (os.cpu_count() or 2)
if shm_gb < 2:
    torch.multiprocessing.set_sharing_strategy('file_system')
if NUM_WORKERS is None:
    NUM_WORKERS = max(0, min(8, cpus - 1)) if shm_gb >= 2 else min(4, max(0, cpus - 1))
print(f'CPU={cpus} | /dev/shm={shm_gb:.2f} GiB | num_workers={NUM_WORKERS}')

## 3. Dữ liệu Food-101 và split
Nếu server không tải được tự động, tải tay **food-101.tar.gz** (~5 GB, md5 `85eeb15f3717b99a5da872d97d918f87`):

- Nguồn gốc ETH: http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz
- Nếu server có internet: mở Terminal của JupyterHub rồi chạy `wget -c -P ~/datasets http://data.vision.ee.ethz.ch/cvl/food-101.tar.gz` (`-c` để tải tiếp khi đứt). Nhanh hơn upload qua trình duyệt.
- Nếu không: tải về máy rồi upload vào `DATA_ROOT`. Upload 5 GB qua JupyterHub dễ đứt; dùng cơ chế chuyển file công ty cho phép.

Đặt file vào `DATA_ROOT/food-101.tar.gz` (hoặc trỏ `FOOD_ARCHIVE` tới đó). Cell kiểm tra md5 rồi giải nén một lần; đã có `DATA_ROOT/food-101/images` thì bỏ qua.

In [ ]:
import hashlib, tarfile
from src.data import load_food101, make_splits

FOOD_MD5 = '85eeb15f3717b99a5da872d97d918f87'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
if not (DATA_ROOT / 'food-101' / 'images').is_dir():
    archive = Path(FOOD_ARCHIVE) if FOOD_ARCHIVE else DATA_ROOT / 'food-101.tar.gz'
    if archive.is_file():
        print(f'Kiểm tra md5 {archive} ({archive.stat().st_size / 2**30:.2f} GiB)...')
        md5 = hashlib.md5()
        with open(archive, 'rb') as f:
            for chunk in iter(lambda: f.read(1 << 24), b''):
                md5.update(chunk)
        assert md5.hexdigest() == FOOD_MD5, 'md5 sai: file tải dở hoặc hỏng, tải lại (wget -c để tải tiếp)'
        print('Giải nén (vài phút)...')
        with tarfile.open(archive) as tar:
            tar.extractall(DATA_ROOT, **({'filter': 'data'} if hasattr(tarfile, 'data_filter') else {}))
    else:
        print('Chưa có archive, thử tải tự động qua torchvision...')
load_food101(str(DATA_ROOT), 'train', download=True)   # đã có thư mục thì không tải lại
n_img = sum(1 for _ in (DATA_ROOT / 'food-101' / 'images').rglob('*.jpg'))
assert n_img == 101000, f'Chỉ thấy {n_img}/101000 ảnh: giải nén chưa đủ'
if not (SPLITS / 'train_idx.json').is_file():
    make_splits(str(DATA_ROOT), str(SPLITS))
print('OK:', DATA_ROOT / 'food-101', '|', n_img, 'ảnh')

## 4. Trạng thái hàng đợi
Thứ tự: R2, R6, R4, R8, R3, R7, R1, R5 (seed 42), sau đó seed 43/44 của các run pretrained.

In [ ]:
import importlib, src.train, src.server
importlib.reload(src.train); importlib.reload(src.server)
from src.server import queue, pending

for run, seed in queue():
    d = OUT / f'{run}_s{seed}'
    if (d / 'metrics.json').is_file():
        state = 'xong'
    elif (d / 'last.pt').is_file():
        state = f'dở, resume ({max(0, sum(1 for _ in open(d / "history.csv")) - 1)} epoch)'
    else:
        state = 'chưa chạy'
    print(f'{run}_s{seed:<4} {state}')

TODO = pending(OUT)        # hoặc tự chọn, ví dụ: TODO = [('R3', 42), ('R7', 42)]
print('\nSẽ chạy:', TODO)

## 5. Train
Chạy trong kernel. Interrupt an toàn; chạy lại cell này để tiếp tục.

In [ ]:
import gc, time
from types import SimpleNamespace
from src.train import run_one, BudgetExhausted

args = SimpleNamespace(data_root=str(DATA_ROOT), out_root=str(OUT), splits_dir=str(SPLITS),
                       num_workers=NUM_WORKERS, epochs=None, lr_backbone=None, batch_size=None,
                       smoke=False, force=False, deadline=time.monotonic() + HOURS * 3600)
try:
    for run, seed in TODO:
        if (OUT / f'{run}_s{seed}' / 'metrics.json').is_file():
            continue
        try:
            run_one(run, seed, args)
        finally:
            gc.collect(); torch.cuda.empty_cache()   # giải phóng model/loader giữa các run
except BudgetExhausted as e:
    print('Hết ngân sách giờ:', e)
except KeyboardInterrupt:
    print('Đã dừng tay. Epoch đã lưu vẫn còn; chạy lại cell để resume.')
print('Còn lại:', pending(OUT))

## 6. Tổng hợp
Chỉ dùng metrics thật (bỏ smoke). Chưa đủ run thì đây là báo cáo tiến độ.

In [ ]:
from IPython.display import display, Markdown, Image
%run -m src.summarize --out_root {OUT} --splits_dir {SPLITS} --fig_dir figures --data_root {DATA_ROOT}
display(Markdown(Path('summary.md').read_text(encoding='utf-8')))
# Hình chính cho báo cáo; bảng LaTeX ở summary.tex, accuracy từng lớp ở per_class.csv.
for name in ['mode_compare', 'curves_by_mode', 'pretrain_gap', 'tradeoff', 'efficiency', 'agreement',
             'per_class', 'confusion_resnet50', 'confusion_deit_s', 'misclassified_resnet50', 'misclassified_deit_s']:
    p = Path('figures') / f'{name}.png'
    if p.exists():
        print(name); display(Image(filename=str(p)))

## 7. Xuất kết quả nhỏ (không có ảnh, .pt, cache)

In [ ]:
import zipfile
from IPython.display import FileLink
EXPORT = A1_DIR / 'exports'; EXPORT.mkdir(exist_ok=True)
archive = EXPORT / 'A1_results.zip'
keep = {'.json', '.yaml', '.csv', '.npy', '.npz', '.png', '.md', '.tex'}
files = [p for folder in [OUT, SPLITS, A1_DIR / 'figures'] for p in folder.rglob('*')
         if p.is_file() and p.suffix in keep and '_smoke' not in str(p) and '_features' not in str(p)]
files += [p for p in map(Path, ['summary.md', 'summary.csv', 'summary.tex', 'per_class.csv', 'runs_all.csv']) if p.exists()]
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, p.resolve().relative_to(A1_DIR.resolve()))
print(f'{archive} ({archive.stat().st_size / 2**20:.1f} MB)')
display(FileLink(os.path.relpath(archive, Path.cwd())))

## 8. (Tuỳ chọn) Đóng gói checkpoint run chưa xong để chuyển máy

In [ ]:
EXPORT_CHECKPOINTS = False
if EXPORT_CHECKPOINTS:
    ck = EXPORT / 'A1_resume_checkpoints.zip'
    with zipfile.ZipFile(ck, 'w', zipfile.ZIP_STORED) as z:
        for d in OUT.glob('R*_s*'):
            if d.name.endswith('_smoke') or (d / 'metrics.json').exists():
                continue
            for name in ['last.pt', 'best.pt', 'config.yaml', 'history.csv']:
                if (d / name).exists():
                    z.write(d / name, (d / name).relative_to(A1_DIR))
    display(FileLink(os.path.relpath(ck, Path.cwd())))